# 1. Busca por apostas e procura por tratamento (Figuras 1 a 5)

Reproduz as seções **"Busca por apostas e procura por tratamento"** e **"Não é só o SUS registrando mais"** do site.

- **Atenção primária (APS):** relatório de produção do SISAB. Atendimentos individuais com o diagnóstico entre os
  problemas avaliados; quem volta várias vezes conta várias vezes.
- **CAPS:** RAAS do SIA-SUS (DATASUS). Pessoas distintas por mês, com o diagnóstico principal ou associado.
- **Controle:** outros transtornos de impulso (F63.1–F63.3: piromania, cleptomania, tricotilomania), sem relação com jogo.
- **Referência de saúde mental na APS:** atendimentos por depressão (F32, F33), ansiedade (F41), estresse (F43),
  álcool (F10) e esquizofrenia (F20).
- Períodos, como no site: APS de jan/2018 a set/2026; CAPS de jan/2018 a jun/2026 (julho e agosto ainda estavam
  incompletos); Google Trends até out/2026 (novembro é parcial).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # para importar src/ (o notebook roda de dentro de notebooks/)

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

from src.config import *
from src.util import *

def faixas_de_contexto(ax):
    """Faixas cinza da pandemia e da CPI das Bets, com o nome acima do gráfico (como no site)."""
    topo = ax.get_xaxis_transform()
    for (ini, fim), cor, nome in ((PANDEMIA, "#F0F0F0", "Pandemia"), (CPI, "#E0E0E0", "CPI das Bets")):
        x0, x1 = pd.to_datetime(ini, format="%Y%m"), pd.to_datetime(fim, format="%Y%m")
        ax.axvspan(x0, x1, color=cor, lw=0)
        ax.text(x0, 1.01, nome, transform=topo, va="bottom", fontsize=7, color="#555555")

def datas(df):
    return pd.to_datetime(df["cmp"].to_list(), format="%Y%m")

def sem_poucos(serie):
    """Meses com menos de 5 ficam em branco."""
    y = serie.to_numpy().astype(float)
    y[y < MIN] = np.nan
    return y

## Dados: atenção primária e CAPS, por mês

In [ ]:
aps = pl.read_parquet(RAW / "atencao_primaria" / "sisab_aps.parquet")
aps = aps.filter((pl.col("recorte") == "total") & pl.col("cmp").is_between(pl.lit(INICIO), pl.lit(FIM_APS)))

def soma(grupos):
    return pl.col("atendimentos").filter(pl.col("grupo").is_in(grupos)).sum()

aps_mes = aps.group_by("cmp").agg(jogo=soma(APS_JOGO), controle=soma(APS_CONTROLE), ref=soma(APS_REF)).sort("cmp")

# CAPS: o diagnóstico F63 da linha é o principal, se for F63, ou senão o associado; contamos pessoas distintas
reg = pl.read_parquet(RAW / "caps" / "sia_f63_registros.parquet")
reg = reg.filter(pl.col("cmp").is_between(pl.lit(INICIO), pl.lit(FIM_CAPS)))
reg = reg.with_columns(
    cid=pl.when(pl.col("CIDPRI").str.starts_with("F63")).then(pl.col("CIDPRI")).otherwise(pl.col("CIDASSOC")))
caps_mes = (reg.group_by("cmp")
            .agg(jogo=pl.col("id").filter(pl.col("cid").is_in(CAPS_JOGO)).n_unique(),
                 controle=pl.col("id").filter(pl.col("cid").is_in(CAPS_CONTROLE)).n_unique())
            .sort("cmp"))

trends = pl.read_parquet(RAW / "exposicao" / "google_trends_uf.parquet").with_columns(cmp=pl.col("data").dt.strftime("%Y%m"))
trends_br = trends.filter((pl.col("uf") == "BR") & pl.col("cmp").is_between(pl.lit(INICIO), pl.lit(FIM_TRENDS)))
aps_mes.tail(3), caps_mes.tail(3)

## Figura 1: atenção primária, atendimentos por mês

In [ ]:
fig, ax = plt.subplots(figsize=(LARGURA, 7 * CM))
faixas_de_contexto(ax)
ax.plot(datas(aps_mes), sem_poucos(aps_mes["controle"]), color=COR["controle_aps"], label="Outros transtornos de impulso (F63.1–F63.3)")
ax.plot(datas(aps_mes), sem_poucos(aps_mes["jogo"]), color=COR["aps"], label="Jogo patológico (F63.0)")
ax.set_ylabel("Atendimentos por mês")
ax.yaxis.set_major_formatter(eixo_br)
ax.legend(loc="upper left")
salvar_figura(fig, "figura1")

## Figura 2: CAPS, pessoas em acompanhamento por mês

In [ ]:
fig, ax = plt.subplots(figsize=(LARGURA, 7 * CM))
faixas_de_contexto(ax)
ax.plot(datas(caps_mes), sem_poucos(caps_mes["controle"]), color=COR["controle_caps"], label="Outros transtornos de impulso (F63.1–F63.3)")
ax.plot(datas(caps_mes), sem_poucos(caps_mes["jogo"]), color=COR["caps"], label="Jogo patológico (F63.0)")
ax.set_ylabel("Pessoas por mês")
ax.yaxis.set_major_formatter(eixo_br)
ax.legend(loc="upper left")
salvar_figura(fig, "figura2")

## Figura 3: busca por apostas no Google

Termos bet, apostas, betano, bet365 e tigrinho. Índice de 0 a 100: mede o interesse relativo, não o número de buscas.

In [ ]:
fig, ax = plt.subplots(figsize=(LARGURA, 6 * CM))
faixas_de_contexto(ax)
ax.plot(datas(trends_br), trends_br["indice"], color=COR["trends"])
ax.set_ylabel("Índice de busca (0–100)")
ax.set_ylim(0, 105)
salvar_figura(fig, "figura3")

## Figura 4: jogo patológico a cada 10 mil atendimentos de saúde mental (APS)

No mesmo mês: atendimentos por F63.0 ÷ atendimentos da referência × 10 mil. Se a alta fosse só mais registro,
essa conta ficaria parada.

In [ ]:
aps_mes = aps_mes.with_columns(por10mil=pl.col("jogo") / pl.col("ref") * 1e4)
y = aps_mes["por10mil"].to_numpy().astype(float)
y[aps_mes["jogo"].to_numpy() < MIN] = np.nan

fig, ax = plt.subplots(figsize=(LARGURA, 7 * CM))
faixas_de_contexto(ax)
ax.plot(datas(aps_mes), y, color=COR["aps"])
ax.set_ylabel("Atendimentos por jogo patológico por\n10 mil atendimentos de saúde mental")
ax.yaxis.set_major_formatter(eixo_br)
salvar_figura(fig, "figura4")

## Figura 5: quantas vezes maior que em 2022

Cada mês dividido pela média mensal de 2022 da mesma série; 1 = igual a 2022 (linha cinza fina). A linha cinza
grossa (outros diagnósticos de saúde mental) mostra até onde o jogo patológico teria subido se a alta fosse só o SUS
registrando mais. O CAPS conta pessoas e a APS conta atendimentos: só o ritmo de crescimento é comparável.

In [ ]:
def vezes_2022(df, col):
    return df[col] / df.filter(pl.col("cmp").str.starts_with("2022"))[col].mean()

fig, ax = plt.subplots(figsize=(LARGURA, 7.5 * CM))
ax.axhline(1, color="#BDBDBD", lw=0.6)
ax.plot(datas(aps_mes), vezes_2022(aps_mes, "jogo"), color=COR["aps"], label="Jogo patológico, atenção primária (atendimentos)")
ax.plot(datas(caps_mes), vezes_2022(caps_mes, "jogo"), color=COR["caps"], label="Jogo patológico, CAPS (pessoas)")
ax.plot(datas(aps_mes), vezes_2022(aps_mes, "ref"), color=COR["ref"], label="Outros diagnósticos de saúde mental, atenção primária")
ax.set_xlim(pd.Timestamp("2022-01-01"), None)
ax.set_ylabel("Vezes a média mensal de 2022")
ax.yaxis.set_major_formatter(eixo_br)
ax.legend(loc="upper left")
salvar_figura(fig, "figura5")

## Números citados no texto

In [ ]:
def media(df, col, ano):
    return df.filter(pl.col("cmp").str.starts_with(str(ano)))[col].mean()

def por10mil(ano):
    x = aps_mes.filter(pl.col("cmp").str.starts_with(str(ano)))
    return x["jogo"].sum() / x["ref"].sum() * 1e4

num = {
    "aps_jogo_mes_2022": br(media(aps_mes, "jogo", 2022), 0),
    "aps_jogo_mes_2025": br(media(aps_mes, "jogo", 2025), 0),
    "aps_jogo_mes_2026": br(media(aps_mes, "jogo", 2026), 0),
    "aps_por10mil_2022": br(por10mil(2022), 1),
    "aps_por10mil_2025": br(por10mil(2025), 1),
    "aps_por10mil_2026": br(por10mil(2026), 1),
    "aps_jogo_vezes_2026_2022": br(media(aps_mes, "jogo", 2026) / media(aps_mes, "jogo", 2022), 0),
    "aps_controle_vezes_2026_2022": br(media(aps_mes, "controle", 2026) / media(aps_mes, "controle", 2022), 1),
    "aps_ref_vezes_2026_2022": br(media(aps_mes, "ref", 2026) / media(aps_mes, "ref", 2022), 1),
    "aps_ultimo_mes_vezes_2022": br(aps_mes["jogo"][-1] / media(aps_mes, "jogo", 2022), 0),
    "caps_jogo_mes_2022": br(media(caps_mes, "jogo", 2022), 0),
    "caps_jogo_mes_2025": br(media(caps_mes, "jogo", 2025), 0),
    "caps_jogo_mes_2026": br(media(caps_mes, "jogo", 2026), 0),
    "caps_controle_vezes_2026_2022": br(media(caps_mes, "controle", 2026) / media(caps_mes, "controle", 2022), 1),
    "caps_ultimo_mes_vezes_2022": br(caps_mes["jogo"][-1] / media(caps_mes, "jogo", 2022), 0),
}
num  # números citados no texto